# Pt1.2 Homework

**Members:**

Martí Serra

Serhii Turtsanash

In [ ]:
y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1]

# Predicciones Modelo A
y_pred_A = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1]

# Predicciones Modelo B
y_pred_B = [1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]

# Predicciones Modelo C
y_pred_C = [1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1]

## 1. Compute metrics

**How is Accuracy computed?**

Accuracy = (TP + TN) / (TP + TN + FP + FN)

It measures the proportion of correct predictions among all predictions.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

models = {'Modelo A': y_pred_A, 'Modelo B': y_pred_B, 'Modelo C': y_pred_C}

print("Resultados:\n")

for name, y_pred in models.items():
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1)
    rec = recall_score(y_true, y_pred, pos_label=1)
    f1 = f1_score(y_true, y_pred, pos_label=1)
    
    print("---", name, "---")
    print("Accuracy:", round(acc, 3))
    print("Precision:", round(prec, 3))
    print("Recall:", round(rec, 3))
    print("F1-Score:", round(f1, 3))
    print("")

## 2. Confusion matrix

In [ ]:
for name, y_pred in models.items():
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    print("---", name, "---")
    print("Matriz de Confusión:")
    print(cm)
    print("")

## 3. Analysis

### Which model has the highest precision? Which has the highest recall?

Highest Precision: Model B (1.000)

Highest Recall: Model C (1.000)

### What error type dominates for each model and why?

Model A: Balanced errors (FP=4, FN=4). This model doesn't favor one error type over the other.

Model B: False Negatives dominate (FN=8, FP=0). It's very conservative, only predicts positive when very confident, missing many actual positives.

Model C: False Positives dominate (FP=8, FN=0). It's very aggressive, predicts positive frequently, catching all positives but also many false alarms.

### Under which application constraints would you pick each model?

Model B (High Precision): Use when false positives are costly. Examples: email spam filtering, fraud detection with automatic blocking.

Model C (High Recall): Use when missing positives is costly. Examples: medical screening, security threats detection.

Model A (Balanced): Use when both error types have similar costs. Example: content moderation.

### How does F1-score reflect the balance between precision and recall? How is f1-score computed? Why not just use simple arithmetic mean?

F1-score formula: F1 = 2 × (Precision × Recall) / (Precision + Recall)

F1-score is the harmonic mean of precision and recall. 

Why not arithmetic mean? Arithmetic mean would be (1.0 + 0.5) / 2 = 0.75, while F1 (harmonic mean) = 2 × (1.0 × 0.5) / 1.5 = 0.67

The harmonic mean penalizes extreme imbalances. If one metric is very low, F1 will be low even if the other is high. This ensures both precision and recall must be reasonably good for a high F1-score.

## 2. NLP with Huggingface transformer library

### Sentiment analysis

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

sa_data = [
    ("Me encanta este producto", "POSITIVE"), ("Es horrible, no lo compren", "NEGATIVE"),
    ("Una maravilla de servicio", "POSITIVE"), ("Llegó roto y tarde", "NEGATIVE"),
    ("Simplemente perfecto", "POSITIVE"), ("Pésima calidad", "NEGATIVE"),
    ("Muy contento con la compra", "POSITIVE"), ("No sirve para nada", "NEGATIVE"),
    ("Lo recomendaría a todos", "POSITIVE"), ("Una estafa total", "NEGATIVE"),
    ("Excelente relación calidad-precio", "POSITIVE"), ("Muy decepcionado", "NEGATIVE"),
    ("Funciona de maravilla", "POSITIVE"), ("El peor dinero gastado", "NEGATIVE"),
    ("Gran experiencia", "POSITIVE"), ("Atención al cliente nula", "NEGATIVE"),
    ("Volveré a comprar", "POSITIVE"), ("Producto defectuoso", "NEGATIVE"),
    ("Me hace feliz", "POSITIVE"), ("Qué desastre", "NEGATIVE")
]

texts = [item[0] for item in sa_data]
y_true_sa = [item[1] for item in sa_data] 

sentiment_pipe = pipeline("sentiment-analysis")

print("Predicciones de sentimiento....")
preds_raw = sentiment_pipe(texts)
y_pred_sa = [p['label'] for p in preds_raw]

acc_sa = accuracy_score(y_true_sa, y_pred_sa)

labels = ["NEGATIVE", "POSITIVE"]
cm_sa = confusion_matrix(y_true_sa, y_pred_sa, labels=labels)

print("Accuracy Sentimiento:", round(acc_sa, 3))
print("Matriz de Confusión:")
print(cm_sa)

**Discussion:**

Accuracy: 80%. The model correctly classified 16/20 samples despite being trained on English and tested on Spanish. The model misclassified 3 negatives as positive and 1 positive as negative. Performance is good considering the language mismatch, but a Spanish-specific model would likely perform better.

### Zero-shot text classification

In [ ]:
zs_data = [
    ("El presidente firmó la nueva ley", "politics"), 
    ("El Real Madrid ganó la copa", "sports"),
    ("Las acciones de Apple subieron", "business"),
    ("El senado debate los presupuestos", "politics"),
    ("Nadal jugará la final", "sports"),
    ("La inflación bajó este mes", "business"),
    ("Nuevas elecciones en primavera", "politics"),
    ("El delantero fichó por otro equipo", "sports"),
    ("La empresa anunció despidos", "business"),
    ("El ministro dará una rueda de prensa", "politics"),
    ("Récord mundial en 100 metros", "sports"),
    ("El mercado de valores cerró al alza", "business"),
    ("Votación decisiva en el parlamento", "politics"),
    ("El partido terminó en empate", "sports"),
    ("Inversión millonaria en tecnología", "business")
]

zs_texts = [item[0] for item in zs_data]
y_true_zs = [item[1] for item in zs_data]
candidate_labels = ["politics", "sports", "business"]

zsc_pipe = pipeline("zero-shot-classification")

print("Realizando clasificación Zero-Shot....")
zs_results = zsc_pipe(zs_texts, candidate_labels=candidate_labels)

y_pred_zs = [res['labels'][0] for res in zs_results]

acc_zs = accuracy_score(y_true_zs, y_pred_zs)
labels_zs = ["business", "politics", "sports"]
cm_zs = confusion_matrix(y_true_zs, y_pred_zs, labels=labels_zs)

print(f"\nAccuracy Zero-Shot (Default): {acc_zs:.3f}")
print(f"Matriz de Confusión ({labels_zs}):\n{cm_zs}")

**Discussion:**

Accuracy: 93.3%. The default model (facebook/bart-large-mnli) performed very well, only misclassifying 1 sports headline as business. All business and politics samples were correctly classified.

In [ ]:
model_name = "joeddav/xlm-roberta-large-xnli"

print("Cargando modelo específico:", model_name, "...")
zsc_better_pipe = pipeline("zero-shot-classification", model=model_name)

zs_results_better = zsc_better_pipe(zs_texts, candidate_labels=candidate_labels)
y_pred_zs_better = [res['labels'][0] for res in zs_results_better]

acc_zs_better = accuracy_score(y_true_zs, y_pred_zs_better)

labels_zs = ["business", "politics", "sports"]
cm_zs_better = confusion_matrix(y_true_zs, y_pred_zs_better, labels=labels_zs)

print("") 
print("Modelo utilizado:", model_name)
print("Accuracy Zero-Shot:", round(acc_zs_better, 3))
print("Matriz de Confusión:")
print(cm_zs_better)

**Discussion:**

Accuracy: 73.3%. The alternative model (xlm-roberta-large-xnli) performed worse than the default. It correctly classified all business and politics samples but struggled with sports (only 1/5 correct). The default model performed better despite not being specifically multilingual.